In [ ]:
import pandas as pd
import os
import re
from pathlib import Path

# Define paths
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
ITERIM_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "interim")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Dir: {RAW_DATA_DIR}")
print(f"Processed Data Dir: {ITERIM_DATA_DIR}")

# Create processed directory if it doesn't exist
os.makedirs(ITERIM_DATA_DIR, exist_ok=True)

Project Root: /Users/rachel/Documents/School/Algonquin College/Business Intelligence/BI2/music-revival-streaming-analysis
Raw Data Dir: /Users/rachel/Documents/School/Algonquin College/Business Intelligence/BI2/music-revival-streaming-analysis/data/raw
Processed Data Dir: /Users/rachel/Documents/School/Algonquin College/Business Intelligence/BI2/music-revival-streaming-analysis/data/iterim


In [2]:
# Cell 2: Helper function to read CSV with encoding detection

def read_csv_safe(file_path):
    """
    Read CSV file with automatic encoding detection
    """
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"  ✓ Successfully read with encoding: {encoding}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  Error with {encoding}: {e}")
            continue
    
    # If all fail, try with errors='ignore'
    print(f"  ⚠ Using fallback encoding with error handling")
    df = pd.read_csv(file_path, encoding='utf-8', errors='ignore')
    return df

print("✓ Safe CSV reader loaded")

✓ Safe CSV reader loaded


In [3]:
# Cell 3: Normalization Function

def normalize_text(text):
    """
    Normalize text for matching:
    - Convert to lowercase
    - Remove punctuation
    - Remove common suffixes (feat, ft, remastered, live, single, etc.)
    - Collapse whitespace
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove common suffixes and patterns
    patterns_to_remove = [
        r'\s*\(feat\.?\s+.*?\)',  # (feat. artist)
        r'\s*\(ft\.?\s+.*?\)',    # (ft. artist)
        r'\s*feat\.?\s+.*?(?=\s|$)',  # feat. artist
        r'\s*ft\.?\s+.*?(?=\s|$)',    # ft. artist
        r'\s*\(remastered.*?\)',   # (remastered ...)
        r'\s*\(live.*?\)',         # (live ...)
        r'\s*\(acoustic.*?\)',     # (acoustic ...)
        r'\s*\(remix.*?\)',        # (remix ...)
        r'\s*\(single.*?\)',       # (single)
        r'\s*-\s*single\s*$',      # - single at end
        r'\s*-\s*remastered\s*$',  # - remastered at end
        r'\s*-\s*live\s*$',        # - live at end
    ]
    
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Remove punctuation (keep spaces and alphanumeric)
    text = re.sub(r'[^\w\s]', '', text)
    
    # Collapse multiple whitespaces into single space
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

print("✓ Normalization function loaded")

✓ Normalization function loaded


In [4]:
def clean_artist_field(x: str) -> str:
    """
    Normalize artist fields that may contain multiple artists or odd formatting.
    - Converts to string
    - Keeps everything, but you could later choose to take only the first artist
    """
    if x is None:
        return ""
    return str(x).strip()

In [ ]:

from utils import extract_main_artist


def standardize_dataset_1(df):
    """
    Standardize Spotify dataset 1 (arnavvvvv/spotify-music)
    Actual columns used:
      - track_name
      - artist(s)_name
    """
    required = ["track_name", "artist(s)_name"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset 1 missing required columns: {missing}. Available: {df.columns.tolist()}")

    df_std = df.copy()

    df_std["track_name_raw"] = df_std["track_name"].astype(str)
    df_std["artist_name_raw"] = df_std["artist(s)_name"].apply(clean_artist_field)

    df_std["track_name_norm"] = df_std["track_name_raw"].apply(normalize_text)
    df_std["artist_name_norm"] = df_std["artist_name_raw"].apply(extract_main_artist).apply(normalize_text)

    df_std["track_artist_key"] = df_std["track_name_norm"] + " | " + df_std["artist_name_norm"]

    return df_std


def standardize_dataset_2(df):
    """
    Standardize Spotify dataset 2 (ambaliyagati/spotify-dataset-for-playing-around-with-sql)
    Actual columns used:
      - name
      - artists
    """
    required = ["name", "artists"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset 2 missing required columns: {missing}. Available: {df.columns.tolist()}")

    df_std = df.copy()

    df_std["track_name_raw"] = df_std["name"].astype(str)
    df_std["artist_name_raw"] = df_std["artists"].apply(clean_artist_field)

    df_std["track_name_norm"] = df_std["track_name_raw"].apply(normalize_text)
    df_std["artist_name_norm"] = df_std["artist_name_raw"].apply(normalize_text)

    df_std["track_artist_key"] = df_std["track_name_norm"] + " | " + df_std["artist_name_norm"]

    return df_std


print("✓ Standardization functions loaded (updated for dataset1 + dataset2 columns)")

✓ Standardization functions loaded (updated for dataset1 + dataset2 columns)


In [6]:
# Cell 5: Load and Standardize All Datasets (UPDATED WITH ENCODING FIX)

def load_and_standardize_all():
    """
    Load both datasets from raw directory and standardize them
    """
    results = {}
    
    # Dataset 1: spotify-music
    dataset1_path = os.path.join(RAW_DATA_DIR, "spotify-music")
    if os.path.exists(dataset1_path):
        print("Processing Dataset 1: spotify-music")
        # Find CSV files in the directory
        csv_files = list(Path(dataset1_path).glob("*.csv"))
        if csv_files:
            try:
                df1 = read_csv_safe(str(csv_files[0]))
                print(f"  Loaded: {csv_files[0].name}")
                print(f"  Shape: {df1.shape}")
                print(f"  Columns: {df1.columns.tolist()}")
                
                df1_std = standardize_dataset_1(df1)
                if df1_std is not None:
                    # Save standardized dataset with UTF-8 encoding
                    output_path1 = os.path.join(ITERIM_DATA_DIR, "spotify_music_standardized.csv")
                    df1_std.to_csv(output_path1, index=False, encoding='utf-8')
                    print(f"  ✓ Standardized and saved to: {output_path1}")
                    results['dataset1'] = df1_std
                else:
                    print("  ✗ Failed to standardize dataset 1")
            except Exception as e:
                print(f"  ✗ Error processing dataset 1: {e}")
        else:
            print(f"  ✗ No CSV files found in {dataset1_path}")
    else:
        print(f"  ✗ Dataset 1 path not found: {dataset1_path}")
    
    # Dataset 2: spotify-dataset
    dataset2_path = os.path.join(RAW_DATA_DIR, "spotify-dataset")
    if os.path.exists(dataset2_path):
        print("\nProcessing Dataset 2: spotify-dataset")
        # Find CSV files in the directory
        csv_files = list(Path(dataset2_path).glob("*.csv"))
        if csv_files:
            try:
                df2 = read_csv_safe(str(csv_files[0]))
                print(f"  Loaded: {csv_files[0].name}")
                print(f"  Shape: {df2.shape}")
                print(f"  Columns: {df2.columns.tolist()}")
                
                df2_std = standardize_dataset_2(df2)
                if df2_std is not None:
                    # Save standardized dataset with UTF-8 encoding
                    output_path2 = os.path.join(ITERIM_DATA_DIR, "spotify_dataset_standardized.csv")
                    df2_std.to_csv(output_path2, index=False, encoding='utf-8')
                    print(f"  ✓ Standardized and saved to: {output_path2}")
                    results['dataset2'] = df2_std
                else:
                    print("  ✗ Failed to standardize dataset 2")
            except Exception as e:
                print(f"  ✗ Error processing dataset 2: {e}")
        else:
            print(f"  ✗ No CSV files found in {dataset2_path}")
    else:
        print(f"  ✗ Dataset 2 path not found: {dataset2_path}")
    
    return results

print("✓ Load and standardize function loaded")

✓ Load and standardize function loaded


In [7]:
# Cell 5: Display Sample Normalization

def display_sample_normalization(df_std, dataset_name, num_samples=5):
    """
    Display sample rows showing raw vs normalized values
    """
    if df_std is None or len(df_std) == 0:
        print(f"No data to display for {dataset_name}")
        return
    
    print("\n" + "="*120)
    print(f"SAMPLE NORMALIZATION RESULTS - {dataset_name}")
    print("="*120)
    
    sample_df = df_std[['track_name_raw', 'artist_name_raw', 'track_name_norm', 'artist_name_norm', 'track_artist_key']].head(num_samples)
    
    for idx, row in sample_df.iterrows():
        print(f"\nRow {idx + 1}:")
        print(f"  Raw Track:      {row['track_name_raw']}")
        print(f"  Norm Track:     {row['track_name_norm']}")
        print(f"  Raw Artist:     {row['artist_name_raw']}")
        print(f"  Norm Artist:    {row['artist_name_norm']}")
        print(f"  Key:            {row['track_artist_key']}")

print("✓ Display function loaded")

✓ Display function loaded


In [8]:
# Cell 6: Run Standardization Pipeline

print("\n" + "="*120)
print("STARTING SPOTIFY DATASET STANDARDIZATION")
print("="*120)

results = load_and_standardize_all()

# Display sample results
if 'dataset1' in results:
    display_sample_normalization(results['dataset1'], "DATASET 1: spotify-music")

if 'dataset2' in results:
    display_sample_normalization(results['dataset2'], "DATASET 2: spotify-dataset")

print("\n" + "="*120)
print("✓ STANDARDIZATION COMPLETE!")
print("="*120)
print(f"\nStandardized datasets saved to: {ITERIM_DATA_DIR}")


STARTING SPOTIFY DATASET STANDARDIZATION
Processing Dataset 1: spotify-music
  ✓ Successfully read with encoding: latin-1
  Loaded: Popular_Spotify_Songs.csv
  Shape: (953, 24)
  Columns: ['track_name', 'artist(s)_name', 'artist_count', 'released_year', 'released_month', 'released_day', 'in_spotify_playlists', 'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm', 'key', 'mode', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%']
  ✓ Standardized and saved to: /Users/rachel/Documents/School/Algonquin College/Business Intelligence/BI2/music-revival-streaming-analysis/data/iterim/spotify_music_standardized.csv

Processing Dataset 2: spotify-dataset
  ✓ Successfully read with encoding: utf-8
  Loaded: spotify_tracks.csv
  Shape: (6300, 8)
  Columns: ['id', 'name', 'genre', 'artists', 'album', 'popularity', 'duration_ms', 'explicit']
  ✓ Sta

In [9]:
# Cell 7: Verify Standardized Files

import glob

print("\n" + "="*120)
print("STANDARDIZED FILES CREATED")
print("="*120)

standardized_files = glob.glob(os.path.join(ITERIM_DATA_DIR, "*_standardized.csv"))

for file in standardized_files:
    df = pd.read_csv(file)
    print(f"\n✓ {os.path.basename(file)}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample keys (first 3):")
    for key in df['track_artist_key'].head(3):
        print(f"    - {key}")


STANDARDIZED FILES CREATED

✓ spotify_dataset_standardized.csv
  Shape: (6300, 13)
  Columns: ['id', 'name', 'genre', 'artists', 'album', 'popularity', 'duration_ms', 'explicit', 'track_name_raw', 'artist_name_raw', 'track_name_norm', 'artist_name_norm', 'track_artist_key']
  Sample keys (first 3):
    - acoustic | billy raffoul
    - acoustic | billy raffoul
    - here comes the sun acoustic | molly hocking bailey rushlow

✓ spotify_music_standardized.csv
  Shape: (953, 29)
  Columns: ['track_name', 'artist(s)_name', 'artist_count', 'released_year', 'released_month', 'released_day', 'in_spotify_playlists', 'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts', 'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts', 'bpm', 'key', 'mode', 'danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%', 'track_name_raw', 'artist_name_raw', 'track_name_norm', 'artist_name_norm', 'track_artist_key']
  Sample keys (f